In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU tersedia: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:

from pathlib import Path
import zipfile

# Lokasi ZIP sesuai hasil pencarian di Google Drive
ZIP_PATH = Path(
    "/content/drive/MyDrive/Dataset Bulai-YOLO/dataset_mobilenet_5kelas_REBUILD.zip"
)

# Lokasi untuk mengekstrak dataset
EXTRACT_DIR = Path("/content/dataset")

# Periksa apakah ZIP tersedia
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"File ZIP tidak ditemukan: {ZIP_PATH}"
    )

# Ekstrak ZIP
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("✅ Dataset berhasil diekstrak!")
print("Lokasi:", EXTRACT_DIR)

print("\nIsi folder:")
for folder in EXTRACT_DIR.iterdir():
    print("-", folder.name)


✅ Dataset berhasil diekstrak!
Lokasi: /content/dataset

Isi folder:
- dataset_mobilenet_5kelas_REBUILD


In [4]:
from pathlib import Path

# Point directly to the extracted folder root
BASE = Path("/content/dataset/dataset_mobilenet_5kelas_REBUILD")

# Verify if the required train, valid, and test folders exist directly under it
required_splits = ["train", "valid", "test"]
all_exist = all((BASE / split).is_dir() for split in required_splits)

if all_exist:
    print("✅ Lokasi dataset terverifikasi:", BASE)
    for split in required_splits:
        split_folder = BASE / split
        subfolders = [p.name for p in split_folder.iterdir() if p.is_dir()]
        print(f"{split}: {len(subfolders)} kelas -> {subfolders}")
else:
    print("❌ Folder train, valid, atau test tidak lengkap di:", BASE)
    print("Isi direktori utama:", [p.name for p in BASE.iterdir() if p.is_dir()])

✅ Lokasi dataset terverifikasi: /content/dataset/dataset_mobilenet_5kelas_REBUILD
train: 5 kelas -> ['Common_Rust', 'Gray_Leaf_Spot', 'Healthy', 'Bulai', 'Blight']
valid: 5 kelas -> ['Common_Rust', 'Gray_Leaf_Spot', 'Healthy', 'Bulai', 'Blight']
test: 5 kelas -> ['Common_Rust', 'Gray_Leaf_Spot', 'Healthy', 'Bulai', 'Blight']


In [5]:

from pathlib import Path

BASE = Path("/content/dataset")

candidates = [
    p for p in BASE.rglob("train")
    if p.is_dir()
    and (p.parent / "valid").is_dir()
    and (p.parent / "test").is_dir()
]

print("Jumlah kandidat:", len(candidates))

for i, train_folder in enumerate(candidates, start=1):
    dataset_folder = train_folder.parent

    print(f"\n{'=' * 40}")
    print(f"KANDIDAT {i}")
    print("Lokasi:", dataset_folder)

    for split in ["train", "valid", "test"]:
        folder = dataset_folder / split
        print(f"\n{split.upper()}")

        for item in sorted(folder.iterdir()):
            if item.is_dir():
                jumlah_gambar = sum(
                    1 for f in item.rglob("*")
                    if f.is_file()
                    and f.suffix.lower() in
                    {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
                )
                print(f"  {item.name}: {jumlah_gambar} gambar")


Jumlah kandidat: 4

KANDIDAT 1
Lokasi: /content/dataset/dataset_mobilenet_5kelas_REBUILD

TRAIN
  Blight: 925 gambar
  Bulai: 1365 gambar
  Common_Rust: 1163 gambar
  Gray_Leaf_Spot: 476 gambar
  Healthy: 1068 gambar

VALID
  Blight: 925 gambar
  Bulai: 1365 gambar
  Common_Rust: 1163 gambar
  Gray_Leaf_Spot: 476 gambar
  Healthy: 1068 gambar

TEST
  Blight: 925 gambar
  Bulai: 1365 gambar
  Common_Rust: 1163 gambar
  Gray_Leaf_Spot: 476 gambar
  Healthy: 1068 gambar

KANDIDAT 2
Lokasi: /content/dataset/dataset_mobilenet_5kelas_REBUILD/train/Bulai

TRAIN
  images: 1190 gambar
  labels: 0 gambar

VALID
  images: 107 gambar
  labels: 0 gambar

TEST
  images: 68 gambar
  labels: 0 gambar

KANDIDAT 3
Lokasi: /content/dataset/dataset_mobilenet_5kelas_REBUILD/test/Bulai

TRAIN
  images: 1190 gambar
  labels: 0 gambar

VALID
  images: 107 gambar
  labels: 0 gambar

TEST
  images: 68 gambar
  labels: 0 gambar

KANDIDAT 4
Lokasi: /content/dataset/dataset_mobilenet_5kelas_REBUILD/valid/Bulai

TR

In [6]:
from pathlib import Path

DATASET_DIR = Path(
    "/content/dataset/dataset_mobilenet_5kelas_REBUILD"
)

CLASSES = [
    "Blight",
    "Bulai",
    "Common_Rust",
    "Gray_Leaf_Spot",
    "Healthy",
]

print("Dataset utama:", DATASET_DIR)
print("Folder tersedia:", DATASET_DIR.exists())

Dataset utama: /content/dataset/dataset_mobilenet_5kelas_REBUILD
Folder tersedia: True


In [7]:

import hashlib

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def get_image_hashes(folder):
    hasil = {}

    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            digest = hashlib.sha256(p.read_bytes()).hexdigest()
            hasil.setdefault(digest, []).append(str(p))

    return hasil

for cls in CLASSES:
    print(f"\n=== {cls} ===")

    split_hashes = {}

    for split in ["train", "valid", "test"]:
        folder = DATASET_DIR / split / cls
        split_hashes[split] = get_image_hashes(folder)

        print(
            f"{split}: {len(split_hashes[split])} gambar unik"
        )

    for a, b in [
        ("train", "valid"),
        ("train", "test"),
        ("valid", "test"),
    ]:
        overlap = (
            set(split_hashes[a])
            & set(split_hashes[b])
        )
        print(f"Identik {a} vs {b}: {len(overlap)}")



=== Blight ===
train: 925 gambar unik
valid: 925 gambar unik
test: 925 gambar unik
Identik train vs valid: 925
Identik train vs test: 925
Identik valid vs test: 925

=== Bulai ===
train: 1314 gambar unik
valid: 1314 gambar unik
test: 1314 gambar unik
Identik train vs valid: 1314
Identik train vs test: 1314
Identik valid vs test: 1314

=== Common_Rust ===
train: 1163 gambar unik
valid: 1163 gambar unik
test: 1163 gambar unik
Identik train vs valid: 1163
Identik train vs test: 1163
Identik valid vs test: 1163

=== Gray_Leaf_Spot ===
train: 476 gambar unik
valid: 476 gambar unik
test: 476 gambar unik
Identik train vs valid: 476
Identik train vs test: 476
Identik valid vs test: 476

=== Healthy ===
train: 1068 gambar unik
valid: 1068 gambar unik
test: 1068 gambar unik
Identik train vs valid: 1068
Identik train vs test: 1068
Identik valid vs test: 1068


In [8]:

from pathlib import Path

BASE = Path("/content/dataset/dataset_mobilenet_5kelas_REBUILD")

for split in ["train", "valid", "test"]:
    print(f"\n=== {split.upper()} ===")

    for cls in [
        "Blight",
        "Bulai",
        "Common_Rust",
        "Gray_Leaf_Spot",
        "Healthy",
    ]:
        folder = BASE / split / cls
        images = [
            p for p in folder.iterdir()
            if p.is_file()
            and p.suffix.lower() in
            {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        ]

        print(f"{cls}: {len(images)} gambar")
        if images:
            print("  Contoh:", images[0].name)



=== TRAIN ===
Blight: 925 gambar
  Contoh: Corn_Blight (841).JPG
Bulai: 0 gambar
Common_Rust: 1163 gambar
  Contoh: Corn_Common_Rust (1066).JPG
Gray_Leaf_Spot: 476 gambar
  Contoh: Corn_Gray_Spot (148).JPG
Healthy: 1068 gambar
  Contoh: Corn_Health (248).jpg

=== VALID ===
Blight: 925 gambar
  Contoh: Corn_Blight (841).JPG
Bulai: 0 gambar
Common_Rust: 1163 gambar
  Contoh: Corn_Common_Rust (1066).JPG
Gray_Leaf_Spot: 476 gambar
  Contoh: Corn_Gray_Spot (148).JPG
Healthy: 1068 gambar
  Contoh: Corn_Health (248).jpg

=== TEST ===
Blight: 925 gambar
  Contoh: Corn_Blight (841).JPG
Bulai: 0 gambar
Common_Rust: 1163 gambar
  Contoh: Corn_Common_Rust (1066).JPG
Gray_Leaf_Spot: 476 gambar
  Contoh: Corn_Gray_Spot (148).JPG
Healthy: 1068 gambar
  Contoh: Corn_Health (248).jpg


In [9]:

from pathlib import Path

BASE = Path("/content/dataset/dataset_mobilenet_5kelas_REBUILD")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Mencari gambar Bulai...\n")

for folder in BASE.rglob("*"):
    if folder.is_dir() and folder.name.lower() == "images":
        if "bulai" in str(folder).lower():
            images = [
                p for p in folder.iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS
            ]
            print("Folder:", folder)
            print("Jumlah gambar:", len(images))
            if images:
                print("Contoh:", images[0].name)
            print()


Mencari gambar Bulai...

Folder: /content/dataset/dataset_mobilenet_5kelas_REBUILD/train/Bulai/valid/images
Jumlah gambar: 107
Contoh: DJI_0659_MP4-0259_jpg.rf.0cdcee765836122008e05934ff9ffa35.jpg

Folder: /content/dataset/dataset_mobilenet_5kelas_REBUILD/train/Bulai/test/images
Jumlah gambar: 68
Contoh: DJI_0659_MP4-0308_jpg.rf.a0fbcd0f1d82dd1b35d7ba14ce4b6d84.jpg

Folder: /content/dataset/dataset_mobilenet_5kelas_REBUILD/train/Bulai/train/images
Jumlah gambar: 1190
Contoh: DJI_0659_MP4-0269_jpg.rf.260a2b796a2515173b706a121628da24.jpg

Folder: /content/dataset/dataset_mobilenet_5kelas_REBUILD/test/Bulai/valid/images
Jumlah gambar: 107
Contoh: DJI_0659_MP4-0259_jpg.rf.0cdcee765836122008e05934ff9ffa35.jpg

Folder: /content/dataset/dataset_mobilenet_5kelas_REBUILD/test/Bulai/test/images
Jumlah gambar: 68
Contoh: DJI_0659_MP4-0308_jpg.rf.a0fbcd0f1d82dd1b35d7ba14ce4b6d84.jpg

Folder: /content/dataset/dataset_mobilenet_5kelas_REBUILD/test/Bulai/train/images
Jumlah gambar: 1190
Contoh: DJI_0

In [10]:

from pathlib import Path
from collections import defaultdict
import hashlib
import random
import shutil

# ==========================================
# 1. LOKASI DATASET
# ==========================================
BASE = Path("/content/dataset/dataset_mobilenet_5kelas_REBUILD")

OUTPUT = Path("/content/dataset_mobilenet_final")

CLASSES = [
    "Blight",
    "Bulai",
    "Common_Rust",
    "Gray_Leaf_Spot",
    "Healthy",
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42

# ==========================================
# 2. FUNGSI HASH UNTUK MENGHINDARI DUPLIKAT
# ==========================================
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# ==========================================
# 3. AMBIL SUMBER GAMBAR SETIAP KELAS
# ==========================================
sources = {
    "Blight": BASE / "train" / "Blight",
    "Common_Rust": BASE / "train" / "Common_Rust",
    "Gray_Leaf_Spot": BASE / "train" / "Gray_Leaf_Spot",
    "Healthy": BASE / "train" / "Healthy",
}

# Foto Bulai berasal dari folder images dataset YOLO.
bulai_root = BASE / "train" / "Bulai"

bulai_sources = [
    bulai_root / "train" / "images",
    bulai_root / "valid" / "images",
    bulai_root / "test" / "images",
]

all_images = {}

for cls in CLASSES:
    if cls == "Bulai":
        folders = bulai_sources
    else:
        folders = [sources[cls]]

    unique = {}
    for folder in folders:
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")

        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                digest = sha256_file(p)
                if digest not in unique:
                    unique[digest] = p

    all_images[cls] = list(unique.values())
    print(f"{cls}: {len(all_images[cls])} gambar unik")

# ==========================================
# 4. BAGI GAMBAR SECARA ACAK
# ==========================================
random.seed(SEED)

splits = {
    "train": {},
    "valid": {},
    "test": {},
}

for cls, images in all_images.items():
    images = images.copy()
    random.shuffle(images)

    n = len(images)
    n_train = int(n * 0.70)
    n_valid = int(n * 0.15)

    splits["train"][cls] = images[:n_train]
    splits["valid"][cls] = images[n_train:n_train + n_valid]
    splits["test"][cls] = images[n_train + n_valid:]

# ==========================================
# 5. SALIN KE FOLDER DATASET BARU
# ==========================================
if OUTPUT.exists():
    raise FileExistsError(
        f"{OUTPUT} sudah ada. Jangan ditimpa otomatis. "
        "Periksa folder tersebut terlebih dahulu."
    )

for split, class_dict in splits.items():
    for cls, images in class_dict.items():
        destination = OUTPUT / split / cls
        destination.mkdir(parents=True, exist_ok=True)

        for i, source in enumerate(images):
            filename = f"{i:05d}_{source.name}"
            shutil.copy2(source, destination / filename)

# ==========================================
# 6. LAPORKAN HASIL
# ==========================================
print("\nDATASET BARU BERHASIL DIBUAT")
print("Lokasi:", OUTPUT)

for split in ["train", "valid", "test"]:
    print(f"\n{split.upper()}")
    for cls in CLASSES:
        folder = OUTPUT / split / cls
        count = sum(
            1 for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        print(f"{cls}: {count} gambar")


Blight: 925 gambar unik
Bulai: 1314 gambar unik
Common_Rust: 1163 gambar unik
Gray_Leaf_Spot: 476 gambar unik
Healthy: 1068 gambar unik

DATASET BARU BERHASIL DIBUAT
Lokasi: /content/dataset_mobilenet_final

TRAIN
Blight: 647 gambar
Bulai: 919 gambar
Common_Rust: 814 gambar
Gray_Leaf_Spot: 333 gambar
Healthy: 747 gambar

VALID
Blight: 138 gambar
Bulai: 197 gambar
Common_Rust: 174 gambar
Gray_Leaf_Spot: 71 gambar
Healthy: 160 gambar

TEST
Blight: 140 gambar
Bulai: 198 gambar
Common_Rust: 175 gambar
Gray_Leaf_Spot: 72 gambar
Healthy: 161 gambar


Langkah 1 — Siapkan dataset untuk MobileNetV2

In [11]:
import tensorflow as tf
from pathlib import Path

DATASET_DIR = Path("/content/dataset_mobilenet_final")

CLASSES = [
    "Blight",
    "Bulai",
    "Common_Rust",
    "Gray_Leaf_Spot",
    "Healthy",
]

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

for split in ["train", "valid", "test"]:
    for cls in CLASSES:
        folder = DATASET_DIR / split / cls

        if not folder.is_dir() or not any(folder.iterdir()):
            raise ValueError(
                f"Folder kosong atau tidak ditemukan: {folder}"
            )

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "train",
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED,
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "valid",
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "test",
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

# Save class_names before wrapping the dataset in prefetch
class_names = train_ds.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Urutan kelas:", class_names)
print("Dataset siap untuk training!")

Found 3460 files belonging to 5 classes.
Found 740 files belonging to 5 classes.
Found 746 files belonging to 5 classes.
Urutan kelas: ['Blight', 'Bulai', 'Common_Rust', 'Gray_Leaf_Spot', 'Healthy']
Dataset siap untuk training!


In [12]:

from tensorflow.keras import layers, Model

# Augmentasi hanya aktif saat training.
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="augmentation")

# Model dasar MobileNetV2 dengan bobot ImageNet.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)

# Bekukan bobot dasar pada tahap pertama.
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3), name="image")

x = augmentation(inputs)

# Preprocessing MobileNetV2: piksel 0–255 menjadi -1 sampai 1.
x = layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1,
    name="mobilenet_preprocessing",
)(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)

outputs = layers.Dense(
    len(CLASSES),
    activation="softmax",
    name="predictions",
)(x)

model = Model(inputs, outputs, name="CornGuard_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "CornGuard_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_preprocessing         │ (None, 224, 224, 3)    │             0 │
│ (Rescaling)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,264,389 (8.64 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:

OUTPUT_DIR = Path("/content/cornguard_mobilenet_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUTPUT_DIR / "mobilenet_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=15,
    callbacks=callbacks,
)

print("Training selesai!")
print("Model terbaik:", OUTPUT_DIR / "mobilenet_best.keras")


Epoch 1/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.6818 - loss: 0.9198
Epoch 1: val_loss improved from None to 0.16202, saving model to /content/cornguard_mobilenet_output/mobilenet_best.keras

Epoch 1: finished saving model to /content/cornguard_mobilenet_output/mobilenet_best.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - accuracy: 0.8350 - loss: 0.4746 - val_accuracy: 0.9486 - val_loss: 0.1620 - learning_rate: 0.0010
Epoch 2/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9400 - loss: 0.1776
Epoch 2: val_loss improved from 0.16202 to 0.13507, saving model to /content/cornguard_mobilenet_output/mobilenet_best.keras

Epoch 2: finished saving model to /content/cornguard_mobilenet_output/mobilenet_best.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 14s 74ms/step - accuracy: 0.9445 - loss: 0.1614 - val_accuracy: 0.9541 - val_loss: 0.1351 - learning_rate: 0.0010
Epoch 3/15
108/109 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.9482 - loss: 0.1380
Epoch 3: val_loss impr

In [14]:

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

best_model = tf.keras.models.load_model(
    OUTPUT_DIR / "mobilenet_best.keras"
)

test_loss, test_accuracy = best_model.evaluate(test_ds)

y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

y_prob = best_model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

print(f"\nTest accuracy: {test_accuracy * 100:.2f}%")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    zero_division=0,
))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASSES))),
))


24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.9678 - loss: 0.0845
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step

Test accuracy: 96.78%

Classification report:
                precision    recall  f1-score   support

        Blight       0.90      0.94      0.92       140
         Bulai       1.00      1.00      1.00       198
   Common_Rust       1.00      0.99      1.00       175
Gray_Leaf_Spot       0.87      0.81      0.83        72
       Healthy       1.00      1.00      1.00       161

      accuracy                           0.97       746
     macro avg       0.95      0.95      0.95       746
  weighted avg       0.97      0.97      0.97       746


Confusion matrix:
[[131   0   0   9   0]
 [  0 198   0   0   0]
 [  1   0 174   0   0]
 [ 14   0   0  58   0]
 [  0   0   0   0 161]]


In [15]:
import os
import tensorflow as tf

folder_model = "/content/cornguard_mobilenet_output"
file_model = os.path.join(folder_model, "mobilenet_best.keras")

print("Versi TensorFlow:", tf.__version__)
print("Folder output tersedia:", os.path.isdir(folder_model))

if os.path.isdir(folder_model):
    print("\nIsi folder output:")
    for nama in os.listdir(folder_model):
        path = os.path.join(folder_model, nama)
        ukuran_mb = os.path.getsize(path) / (1024 * 1024) if os.path.isfile(path) else 0
        print(f"- {nama} ({ukuran_mb:.2f} MB)" if os.path.isfile(path) else f"- {nama}/")

print("\nFile model terbaik tersedia:", os.path.isfile(file_model))

Versi TensorFlow: 2.21.0
Folder output tersedia: True

Isi folder output:
- mobilenet_best.keras (9.26 MB)

File model terbaik tersedia: True


In [16]:
import os
import tensorflow as tf

model_path = "/content/cornguard_mobilenet_output/mobilenet_best.keras"

model = tf.keras.models.load_model(model_path, compile=False)

print("Input model :", model.input_shape)
print("Output model:", model.output_shape)
print("Nama model  :", model.name)

print("\nLayer preprocessing:")
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Rescaling):
        print(
            "Layer:", layer.name,
            "| scale:", layer.scale,
            "| offset:", layer.offset
        )

print("\nDaftar kelas dari variabel training:")
if "CLASSES" in globals():
    print(CLASSES)
else:
    print("Variabel CLASSES tidak ditemukan di sesi ini.")

Input model : (None, 224, 224, 3)
Output model: (None, 5)
Nama model  : CornGuard_MobileNetV2

Layer preprocessing:
Layer: mobilenet_preprocessing | scale: 0.00784313725490196 | offset: -1

Daftar kelas dari variabel training:
['Blight', 'Bulai', 'Common_Rust', 'Gray_Leaf_Spot', 'Healthy']


In [17]:
%pip install -q tf2onnx onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.4 MB/s eta 0:00:00


In [18]:
import os
import tensorflow as tf
import tf2onnx
import onnx

model_path = "/content/cornguard_mobilenet_output/mobilenet_best.keras"
output_path = "/content/cornguard_mobilenet_output/classifier_5kelas.onnx"

model = tf.keras.models.load_model(model_path, compile=False)

input_signature = (
    tf.TensorSpec(
        shape=(None, 224, 224, 3),
        dtype=tf.float32,
        name="image"
    ),
)

onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=13,
    output_path=output_path
)

onnx.checker.check_model(onnx_model)

print("Ekspor ONNX berhasil!")
print("Lokasi:", output_path)
print("Ukuran: {:.2f} MB".format(os.path.getsize(output_path) / (1024 * 1024)))
print("Input ONNX:", onnx_model.graph.input[0].name)
print("Output ONNX:", onnx_model.graph.output[0].name)

Ekspor ONNX berhasil!
Lokasi: /content/cornguard_mobilenet_output/classifier_5kelas.onnx
Ukuran: 8.53 MB
Input ONNX: image
Output ONNX: predictions


In [19]:
import numpy as np
import onnxruntime as ort

onnx_path = "/content/cornguard_mobilenet_output/classifier_5kelas.onnx"

session = ort.InferenceSession(
    onnx_path,
    providers=["CPUExecutionProvider"]
)

input_info = session.get_inputs()[0]
output_info = session.get_outputs()[0]

print("Input shape :", input_info.shape)
print("Output shape:", output_info.shape)

dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
scores = session.run(
    [output_info.name],
    {input_info.name: dummy}
)[0]

print("Bentuk hasil prediksi:", scores.shape)
print("Jumlah skor kelas:", scores.shape[-1])
print("Nilai output:", scores)

Input shape : ['unk__654', 224, 224, 3]
Output shape: ['unk__655', 5]
Bentuk hasil prediksi: (1, 5)
Jumlah skor kelas: 5
Nilai output: [[0.01481175 0.00943835 0.4648895  0.00075954 0.5101009 ]]


In [20]:
from google.colab import files

files.download(
    "/content/cornguard_mobilenet_output/classifier_5kelas.onnx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import json
from google.colab import files

# 1. Sesuaikan urutan label kelas sesuai urutan dataset kamu
classes_list = ["Blight", "Common_Rust", "Gray_Leaf_Spot", "Healthy", "Bulai"]

# 2. Buat struktur data dictionary
metadata = {
    "input_size": 224,
    "classes": classes_list,
    "preprocess": "none",
    "resize": "stretch"
}

# 3. Simpan ke file JSON
json_filename = "classifier_meta.json"
with open(json_filename, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"File {json_filename} berhasil dibuat!")

# 4. Otomatis download file ke laptop
files.download(json_filename)

File classifier_meta.json berhasil dibuat!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# Membuat pemetaan indeks kelas dari list CLASSES yang telah didefinisikan sebelumnya
class_indices = {name: idx for idx, name in enumerate(CLASSES)}
print(class_indices)

{'Blight': 0, 'Bulai': 1, 'Common_Rust': 2, 'Gray_Leaf_Spot': 3, 'Healthy': 4}
